# 推理 Infra 学习实录 02：nano-vLLM vs Transformers

> 归档状态（GitHub 发布副本）：历史手记数据，无原始运行输出或原始 JSON，待复核；预热与采样口径不统一，不能用于框架排名。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../01/article.md) · [下一篇](../03/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 用来配套第二篇文章：**nano-vLLM vs Transformers：同一个 Qwen 模型，推理引擎到底多做了什么**。

目标不是做严格生产 benchmark，而是在同一个魔搭 Notebook 环境里，用同一个 Qwen 小模型、同一组 prompt，对比两条生成路径：

```text
Transformers: tokenizer -> model.generate() -> decode
nano-vLLM: LLMEngine -> Scheduler -> ModelRunner -> KV Cache -> Sampler
```

本 Notebook 会记录：

- 总耗时
- 输出 token 数
- tokens/s
- 峰值显存
- 输出样例

注意：为了避免两套框架在同一个进程里抢显存，建议按顺序运行，并在跑完 Transformers 后执行显存清理单元。

## 1. 环境检查

In [ ]:
!nvidia-smi

In [ ]:
import sys

print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", repr(e))

## 2. 安装依赖

如果环境里已经安装过这些包，可以跳过本节。

nano-vLLM 依赖 `flash-attn`、`triton`、CUDA 版本和 PyTorch 版本，安装失败时通常需要先检查环境组合。

`pandas` 用于表格，`device_map="auto"` 需要 `accelerate`。已安装 nano-vLLM 时也应核对固定 commit，而不是仅凭包名跳过版本检查。

In [ ]:
# 如未安装，取消注释运行；device_map="auto" 需要 accelerate
# %pip install modelscope pandas accelerate

In [ ]:
# 如未安装 nano-vLLM，取消注释运行
# %pip install git+https://github.com/GeeeekExplorer/nano-vllm.git@bb823b3e06983d71485a8e1f23715ebd87d98ef8

## 3. 下载模型

本次使用 `Qwen/Qwen3-0.6B`。如果模型已经下载，可以直接把 `model_dir` 改成已有路径。

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)

print("model_dir =", model_dir)

In [ ]:
import os

print("模型目录:", model_dir)
print("目录内容:")
for name in os.listdir(model_dir)[:30]:
    print(" -", name)

## 4. 准备同一组 prompt

两套框架使用同一组原始 prompt，并用同一个 tokenizer 的 chat template 包装。

In [ ]:
from transformers import AutoTokenizer

raw_prompts = [
    "请用三句话解释什么是大模型推理引擎。",
    "请解释 KV Cache 在大模型推理中的作用。",
    "What is the difference between prefill and decode in LLM inference?",
    "如果有 100 个用户同时请求大模型服务，推理系统需要处理哪些问题？",
]

tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

chat_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for prompt in raw_prompts
]

print("prompt 数:", len(chat_prompts))
print(chat_prompts[0][:500])

## 5. 公共统计函数

显存除以 `1024**3`，单位是 GiB。`peak_memory_gb` 暂保留旧字段名，展示为 `caller_peak_allocated_gib`：它只覆盖调用进程的 PyTorch allocator，不一定覆盖 worker，也不是引擎总显存。nano-vLLM 初始化时预分配 KV Cache，不能把预留容量解读为本轮请求的增量显存。

In [ ]:
import gc
import time
import pandas as pd
import torch


def reset_cuda_stats():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def current_peak_memory_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024**3


def cleanup_cuda(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


results = []

## 6. Transformers 测试

Transformers 路径可以这样理解：

```text
tokenizer -> AutoModelForCausalLM -> model.generate() -> decode
```

In [ ]:
from transformers import AutoModelForCausalLM

tokenizer_tf = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
tokenizer_tf.padding_side = "left"
if tokenizer_tf.pad_token_id is None:
    tokenizer_tf.pad_token = tokenizer_tf.eos_token

model_tf = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype="auto",
    device_map="auto",
)
model_tf.eval()

inputs_tf = tokenizer_tf(
    chat_prompts,
    return_tensors="pt",
    padding=True,
).to(model_tf.device)

reset_cuda_stats()

t0 = time.perf_counter()
with torch.inference_mode():
    output_ids_tf = model_tf.generate(
        **inputs_tf,
        max_new_tokens=256,
        temperature=0.6,
        do_sample=True,
        pad_token_id=tokenizer_tf.pad_token_id,
        eos_token_id=tokenizer_tf.eos_token_id,
    )
torch.cuda.synchronize()
elapsed_tf = time.perf_counter() - t0

input_len_tf = inputs_tf["input_ids"].shape[1]
new_token_ids_tf = output_ids_tf[:, input_len_tf:]
output_tokens_tf = int((new_token_ids_tf != tokenizer_tf.pad_token_id).sum().item())
tokens_per_second_tf = output_tokens_tf / elapsed_tf
peak_memory_tf = current_peak_memory_gb()
texts_tf = tokenizer_tf.batch_decode(new_token_ids_tf, skip_special_tokens=True)

transformers_result = {
    "engine": "Transformers",
    "num_prompts": len(chat_prompts),
    "max_new_tokens": 256,
    "elapsed_s": elapsed_tf,
    "output_tokens": output_tokens_tf,
    "tokens_per_second": tokens_per_second_tf,
    "peak_memory_gb": peak_memory_tf,
}

results.append(transformers_result)
transformers_result

In [ ]:
for i, text in enumerate(texts_tf):
    print("=" * 80)
    print("Prompt", i)
    print(raw_prompts[i])
    print("-" * 80)
    print(text[:1200])

## 7. 清理 Transformers 显存

如果不清理，后面初始化 nano-vLLM 时可能出现显存不足。

In [ ]:
del model_tf
del inputs_tf
del output_ids_tf
del new_token_ids_tf
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("CUDA memory allocated GiB:", torch.cuda.memory_allocated() / 1024**3)
print("CUDA memory reserved GiB:", torch.cuda.memory_reserved() / 1024**3)

## 8. nano-vLLM 测试

nano-vLLM 路径可以这样理解：

```text
LLMEngine -> Scheduler -> ModelRunner -> Attention / KV Cache -> Sampler
```

这里先使用 `enforce_eager=True`，减少 CUDA Graph 对入门观察的影响。后续可以单独测试 `enforce_eager=False`。

重跑或切换 eager/graph 时先结束旧进程或重启 kernel，不要在同一进程重复创建 LLM。严格对照应采用第 07 篇的独立环境、独立进程方案。

In [ ]:
from nanovllm import LLM, SamplingParams

llm = LLM(
    model_dir,
    enforce_eager=True,
    tensor_parallel_size=1,
)

sampling_params = SamplingParams(
    temperature=0.6,
    max_tokens=256,
)

reset_cuda_stats()

t0 = time.perf_counter()
outputs_nv = llm.generate(chat_prompts, sampling_params)
torch.cuda.synchronize()
elapsed_nv = time.perf_counter() - t0

output_tokens_nv = sum(len(output["token_ids"]) for output in outputs_nv)
tokens_per_second_nv = output_tokens_nv / elapsed_nv
peak_memory_nv = current_peak_memory_gb()

nanovllm_result = {
    "engine": "nano-vLLM",
    "num_prompts": len(chat_prompts),
    "max_new_tokens": 256,
    "elapsed_s": elapsed_nv,
    "output_tokens": output_tokens_nv,
    "tokens_per_second": tokens_per_second_nv,
    "peak_memory_gb": peak_memory_nv,
}

results.append(nanovllm_result)
nanovllm_result

In [ ]:
for i, output in enumerate(outputs_nv):
    print("=" * 80)
    print("Prompt", i)
    print(raw_prompts[i])
    print("-" * 80)
    print(output["text"][:1200])

## 9. 汇总对比

In [ ]:
df = pd.DataFrame(results)
df.rename(columns={"peak_memory_gb": "caller_peak_allocated_gib"})

In [ ]:
summary = df.copy()
summary["elapsed_s"] = summary["elapsed_s"].map(lambda x: round(x, 4))
summary["tokens_per_second"] = summary["tokens_per_second"].map(lambda x: round(x, 4))
summary["peak_memory_gb"] = summary["peak_memory_gb"].map(lambda x: round(x, 4))
summary.rename(columns={"peak_memory_gb": "caller_peak_allocated_gib"})

以下为新运行记录模板，不自动覆盖历史手记表。正文历史数据无原始运行输出或 JSON，预热与采样未统一，待复核且不可排名：

| 指标 | Transformers | nano-vLLM |
|---|---:|---:|
| prompt 数 | 4 | 4 |
| max new tokens | 256 | 256 |
| 总输出 token | 运行后填写 | 运行后填写 |
| 总耗时 | 运行后填写 | 运行后填写 |
| tokens/s | 运行后填写 | 运行后填写 |
| 调用进程 peak allocated（GiB） | 运行后填写 | 运行后填写 |

这组数据不用于给框架排名。它更适合用来观察两种路径的系统差异：

```text
Transformers 更像模型调用入口。
nano-vLLM 更像推理系统入口。
```

## 10. 可选实验：更大的 batch

如果显存允许，可以构造更多 prompt 来观察 batch 变大时的表现。

注意：这个实验会占用更多显存，建议先确认前面的实验已经跑通。

In [ ]:
# 可选：构造更多请求，重新跑 nano-vLLM 或 Transformers
# batch_prompts = chat_prompts * 8
# len(batch_prompts)

## 11. 本次实验应该观察什么

跑完后，不要只看谁更快。更重要的是观察：

- Transformers 的代码路径是否更像一次模型调用。
- nano-vLLM 的进度条是否区分了 Prefill 和 Decode。
- 小 batch 下 nano-vLLM 是否不一定占优。
- 显存峰值是否和初始化、KV Cache 预分配有关。
- 如果增大 batch，两者行为是否开始出现更明显差异。

这正好对应第二篇文章的核心问题：

```text
当一次生成从“单用户调用”变成“多请求服务化生成”时，
系统复杂度从哪里冒出来？
```